# Lab 3.5 &mdash; Cycles and Retry

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; LangGraph: Stateful Agent Workflows**

### What you'll do
- Build a loop &mdash; an edge that points backwards, and nothing more exotic
- Give it a step budget, because the runaway loop is Module 2's failure mode
- Put the model inside <i>one</i> node and leave the control flow deterministic

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All eight Module 3 labs work one case: leave requests in a small HR
> system. The rules are ordinary on purpose &mdash; the only new thing here is LangGraph.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# Leave requests in a small HR system. Ordinary rules on purpose: the only new thing in these
# eight labs is LangGraph. One flat dict -- no joins, no helpers, nothing to learn here.

REQUESTS = {
    "LV-5001": {"who": "Priya Nair",   "days":  3, "kind": "annual", "reason": "family wedding",
                "balance": 12, "manager": "Devi R."},
    "LV-5002": {"who": "Rahul Menon",  "days":  5, "kind": "annual", "reason": "",
                "balance":  3, "manager": "Devi R."},
    "LV-5003": {"who": "Anita Sharma", "days":  2, "kind": "annual", "reason": "moving house",
                "balance":  0, "manager": "Sam O."},
    "LV-5004": {"who": "Vikram Rao",   "days": 15, "kind": "annual", "reason": "sabbatical",
                "balance": 20, "manager": "Sam O."},
    "LV-5005": {"who": "Priya Nair",   "days":  1, "kind": "sick",   "reason": "flu",
                "balance": 12, "manager": "Devi R."},
}

# The handbook, as three numbers. Every routing decision in this module comes from these.
POLICY = {"manager_over_days": 2, "hr_over_days": 10, "max_clarifications": 2}

print(len(REQUESTS), "leave requests loaded")

## Concept

A cycle is an edge that points backwards. There is no loop construct in LangGraph &mdash;
`add_edge("ask", "validate")` and you have one.

So the interesting part is not building the loop, it is **stopping** it. In a graph you write that
down as a field and a comparison.

`LV-5002` has an empty `reason`. Go back to the employee, ask, try again &mdash; but not forever.

```
              +-----------------+
              v                 |
START -> validate -> (route) -> ask
                       |
                       +--> decide     (it is complete)
                       +--> withdraw   (we asked enough times)
```

## Section 1 &mdash; The router, and the budget

Three ways out of `validate`. `POLICY` has three numbers in it; one is how many times we are
willing to go back.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END


class ReviewState(TypedDict):
    request_id: str
    reason: str
    complete: bool
    attempts: int                   # replaced, not accumulated -- it is a counter
    inbox: dict                     # what the employee replies, keyed by attempt number
    decision: str
    notes: Annotated[list, add]


def route_after_validate(state: ReviewState) -> str:
    """complete -> decide.  incomplete with budget left -> ask.  budget spent -> withdraw."""
    if state["complete"]:
        return "decide"

    if state["attempts"] >= POLICY["max_clarifications"]:
        return "withdraw"                       # the budget is what makes the cycle terminate

    return "ask"

In [ ]:
# --- Self-check: Section 1
def r(complete, attempts):
    return route_after_validate({"complete": complete, "attempts": attempts})

check("the budget is 2, so a third pass withdraws instead of asking again",
      lambda: r(False, 0) == "ask" and r(False, 1) == "ask" and r(False, 2) == "withdraw",
      ">= is what makes it a budget rather than a suggestion")
check("a complete request is never withdrawn, however long it took",
      lambda: r(True, 0) == "decide" and r(True, 9) == "decide",
      "check completeness first -- getting the answer late is still getting the answer")
score()

## Section 2 &mdash; The backward edge

`ask` bumps the counter and reads whatever the employee sent back. Then one edge returns the run
to `validate` &mdash; and that edge is the cycle.

In [ ]:
def validate(state: ReviewState) -> dict:
    reason = state["reason"] or REQUESTS[state["request_id"]]["reason"]
    return {"reason": reason, "complete": bool(reason.strip()),
            "notes": [f'validate(attempt={state["attempts"]}, complete={bool(reason.strip())})']}

def ask_employee(state: ReviewState) -> dict:
    n = state["attempts"] + 1
    return {"attempts": n, "reason": state["inbox"].get(n, ""),
            "notes": [f'ask(attempt={n})']}

def decide(state: ReviewState) -> dict:
    return {"decision": f'accepted: {state["reason"]}', "notes": ["decide"]}

def withdraw(state: ReviewState) -> dict:
    return {"decision": f'withdrawn after {state["attempts"]} request(s)', "notes": ["withdraw"]}


def build_loop(ask_node=ask_employee):
    builder = StateGraph(ReviewState)
    builder.add_node("validate", validate)
    builder.add_node("ask", ask_node)
    builder.add_node("decide", decide)
    builder.add_node("withdraw", withdraw)

    builder.add_edge(START, "validate")
    builder.add_conditional_edges("validate", route_after_validate,
                                  {"ask": "ask", "decide": "decide", "withdraw": "withdraw"})

    builder.add_edge("ask", "validate")   # the cycle: go round and check again

    builder.add_edge("decide", END)
    builder.add_edge("withdraw", END)
    return builder.compile()

In [ ]:
# --- Self-check: Section 2   (a real cycle that really terminates -- no model)
BASE = {"reason": "", "complete": False, "attempts": 0, "inbox": {}, "decision": "", "notes": []}

def run(rid, inbox=None, ask_node=ask_employee):
    return build_loop(ask_node).invoke({**BASE, "request_id": rid, "inbox": inbox or {}})

check("LV-5001 already has a reason, so the loop never runs",
      lambda: run("LV-5001")["decision"].startswith("accepted")
          and not any("ask(" in n for n in run("LV-5001")["notes"]))
check("LV-5002 gets asked, replies on attempt 2, and the loop ends",
      lambda: run("LV-5002", {2: "conference in Berlin"})["decision"].startswith("accepted"),
      "validate has to run again to notice the reply -- that is the backward edge")
check("with no reply ever, the budget stops it rather than looping forever",
      lambda: run("LV-5002")["decision"].startswith("withdrawn")
          and run("LV-5002")["attempts"] == POLICY["max_clarifications"],
      "this is the check that would hang if the budget were missing")
score()

## Watch it run

Both exits. The repeated `validate` lines are the cycle.

In [ ]:
if guard(build_loop) is not None:
    for label, inbox in [("replies on attempt 2", {2: "conference in Berlin"}),
                         ("never replies", {})]:
        out = run("LV-5002", inbox)
        print(f"=== LV-5002, {label} ===")
        for n in out["notes"]:
            print("   ", n)
        print("    ->", out["decision"], "\n")

## Run it for real &mdash; the model inside one node

Same graph. `ask_llm` has the model write the message to the employee, and changes nothing else.
That is the pattern to take away: **the model is a node, not the architecture.** The routing, the
budget and the cycle stay ordinary code you can test offline.

In [ ]:
def ask_llm(state: ReviewState) -> dict:
    n = state["attempts"] + 1
    r = REQUESTS[state["request_id"]]
    msg = ask(f'Ask {r["who"]} to give a reason for their {r["days"]}-day {r["kind"]} '
                    f'leave request. One short polite sentence, no greeting, no sign-off.',
                    system="You write brief internal HR messages. Plain text only.")
    return {"attempts": n, "reason": state["inbox"].get(n, ""),
            "notes": [f'ask({n}): {msg.strip()[:100]}']}


def live_run():
    out = run("LV-5002", {2: "conference in Berlin"}, ask_node=ask_llm)
    for n in out["notes"]:
        print("   ", n)
    print("    ->", out["decision"])

if llm_ready():
    guard(live_run)

### Read it

The run went round the cycle exactly as many times as the deterministic one, and stopped for the
same reason. The model wrote a sentence; it decided nothing.

Nothing about the loop got less testable: the Section 2 checks still run offline and still cover
the budget &mdash; the one behaviour you cannot afford to have flake. Lab 1.1 spun in a `while True`
for want of exactly this, and here the same mistake is a missing `>=` in a four-line function.

In [ ]:
score()

## Your turn

1. Set `POLICY["max_clarifications"]` to 0 and re-run. Does it ask once, or not at all? Read the
   router and decide before you run it.
2. Remove your budget check and pass `{"recursion_limit": 4}` to `invoke`. Which error would you
   rather explain &mdash; "withdrawn after 2 requests" or `GraphRecursionError`?